# Extracción de imágenes del dataset
En el siguiente documento se lleva a cabo un procesamiento del pdf que contiene el dataset, extrayendo página por página todas las imágenes, añadiendo la etiqueta correspondiente y limpiándolas de ruido y binarizando la imagen.


Instalación de librerías

In [6]:
pip install pymupdf opencv-python numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 49.1 MB/s eta 0:00:00


Extracción en baja calidad

In [ ]:
import fitz  # PyMuPDF
import cv2
import numpy as np
import os

def ordenar_cajas(cajas, max_y_diff=50):
    if not cajas:
        return []
    cajas = sorted(cajas, key=lambda b: b[1])
    filas = []
    fila_actual = [cajas[0]]
    for caja in cajas[1:]:
        if abs(caja[1] - fila_actual[0][1]) < max_y_diff:
            fila_actual.append(caja)
        else:
            fila_actual = sorted(fila_actual, key=lambda b: b[0])
            filas.extend(fila_actual)
            fila_actual = [caja]
    fila_actual = sorted(fila_actual, key=lambda b: b[0])
    filas.extend(fila_actual)
    return filas

def extraer_imagenes_de_pdf(pdf_path, vector_nombres, output_dir="dataset_aislado", tamano_final=(400, 200)):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    doc = fitz.open(pdf_path)

    if len(vector_nombres) != len(doc):
        print(f"Aviso: El PDF tiene {len(doc)} páginas y el vector tiene {len(vector_nombres)} elementos.")

    contador_ceros = 0

    for num_pagina in range(min(len(doc), len(vector_nombres))):

        # --- LÓGICA DE NOMENCLATURA ACTUALIZADA ---
        prefijo_base = vector_nombres[num_pagina]
        if prefijo_base == 0:
            contador_ceros += 1
            prefijo = f"00_{contador_ceros:02d}"
        else:
            # Rellena con un cero a la izquierda si es de 1 cifra (ej: 1 -> "01", 15 -> "15")
            prefijo = f"{prefijo_base:02d}"

        pagina = doc.load_page(num_pagina)
        pix = pagina.get_pixmap(dpi=200)

        img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.h, pix.w, pix.n)

        if pix.n == 4:
            img = cv2.cvtColor(img, cv2.COLOR_RGBA2BGR)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

        # ... (código anterior igual) ...
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # 1. Umbral más permisivo (150 en lugar de 100) para captar trazos más suaves
        _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY_INV)

        # 2. Engrosamiento (Dilatación) de las líneas finas para que no se rompan
        kernel = np.ones((3, 3), np.uint8)
        thresh = cv2.dilate(thresh, kernel, iterations=1)

        contornos, _ = cv2.findContours(thresh, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)

        area_total_pagina = img.shape[0] * img.shape[1]
        cajas_brutas = []

        for c in contornos:
            x, y, w, h = cv2.boundingRect(c)
            area = w * h
            # 3. Reducción del área mínima de 5000 a 3000
            if 3000 < area < (area_total_pagina * 0.5):
                cajas_brutas.append((x, y, w, h))
        # ... (el resto del código continúa igual) ...

        cajas_unicas = []
        for caja in cajas_brutas:
            x, y, w, h = caja
            es_duplicada = False
            for (ux, uy, uw, uh) in cajas_unicas:
                if abs(x - ux) < 30 and abs(y - uy) < 30:
                    es_duplicada = True
                    break
            if not es_duplicada:
                cajas_unicas.append(caja)

        cajas_unicas.sort(key=lambda b: b[2] * b[3], reverse=True)
        doce_cajas = cajas_unicas[:12]

        cajas_ordenadas = ordenar_cajas(doce_cajas)

        MARGEN_BORDE = 20

        for i, (x, y, w, h) in enumerate(cajas_ordenadas):
            if h > 2 * MARGEN_BORDE and w > 2 * MARGEN_BORDE:
                recorte = img[y + MARGEN_BORDE : y + h - MARGEN_BORDE, x + MARGEN_BORDE : x + w - MARGEN_BORDE]
            else:
                recorte = img[y:y+h, x:x+w]

            recorte_gris = cv2.cvtColor(recorte, cv2.COLOR_BGR2GRAY)
            recorte_suavizado = cv2.GaussianBlur(recorte_gris, (5, 5), 0)
            _, recorte_limpio = cv2.threshold(recorte_suavizado, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

            recorte_invertido = cv2.bitwise_not(recorte_limpio)
            puntos_tinta = cv2.findNonZero(recorte_invertido)

            if puntos_tinta is not None:
                x_f, y_f, w_f, h_f = cv2.boundingRect(puntos_tinta)
                firma_ajustada = recorte_limpio[y_f:y_f+h_f, x_f:x_f+w_f]
                firma_con_margen = cv2.copyMakeBorder(firma_ajustada, 10, 10, 10, 10, cv2.BORDER_CONSTANT, value=255)
            else:
                firma_con_margen = recorte_limpio

            h_actual, w_actual = firma_con_margen.shape
            w_final, h_final = tamano_final

            if h_actual == 0 or w_actual == 0:
                continue

            escala = min(w_final / w_actual, h_final / h_actual)
            nuevo_w = int(w_actual * escala)
            nuevo_h = int(h_actual * escala)

            firma_redimensionada = cv2.resize(firma_con_margen, (nuevo_w, nuevo_h), interpolation=cv2.INTER_AREA)
            lienzo_blanco = np.ones((h_final, w_final), dtype=np.uint8) * 255

            x_offset = (w_final - nuevo_w) // 2
            y_offset = (h_final - nuevo_h) // 2
            lienzo_blanco[y_offset:y_offset+nuevo_h, x_offset:x_offset+nuevo_w] = firma_redimensionada

            # --- CAMBIO SOLICITADO: Inclusión del número de página al inicio ---
            # Formato: PPP_PRE_II.png (PPP: página, PRE: prefijo, II: índice imagen)
            nombre_archivo = f"{num_pagina + 1:03d}_{prefijo}_{i + 1:02d}.png"

            ruta_archivo = os.path.join(output_dir, nombre_archivo)
            cv2.imwrite(ruta_archivo, lienzo_blanco)

        print(f"Página {num_pagina + 1} procesada. Prefijo base usado: '{prefijo}'.")

    print("¡Proceso completado!")

# --- EJECUCIÓN DEL SCRIPT ---
if __name__ == "__main__":
    from google.colab import drive
    drive.mount('/content/drive')
    ruta_al_pdf = "/content/drive/MyDrive/CUARTO/TFG/DATASET/DATASET.pdf"

    vector_de_nombres =[0,1,0,0,2,3,4,5,6,7,8,9,10,11,12,13,14,15,0,0,0,0,0,16,17,0,0,0,18,19,20,21,22,23,24,25,0,0,0,0,0,0,0,0,0,0,26,27,28,0,0,0,0,29,0,0,30,0,0,0,0,0,0,31,32,33,34,35,36,37,38,39,40,41,42,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]

    TAMAÑO_UNIFORME = (800, 600)

    extraer_imagenes_de_pdf(
        pdf_path=ruta_al_pdf,
        vector_nombres=vector_de_nombres,
        output_dir="IMAGENES_LOW",
        tamano_final=TAMAÑO_UNIFORME
    )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Página 1 procesada. Prefijo base usado: '00_01'.
Página 2 procesada. Prefijo base usado: '01'.
Página 3 procesada. Prefijo base usado: '00_02'.
Página 4 procesada. Prefijo base usado: '00_03'.
Página 5 procesada. Prefijo base usado: '02'.
Página 6 procesada. Prefijo base usado: '03'.
Página 7 procesada. Prefijo base usado: '04'.
Página 8 procesada. Prefijo base usado: '05'.
Página 9 procesada. Prefijo base usado: '06'.
Página 10 procesada. Prefijo base usado: '07'.
Página 11 procesada. Prefijo base usado: '08'.
Página 12 procesada. Prefijo base usado: '09'.
Página 13 procesada. Prefijo base usado: '10'.
Página 14 procesada. Prefijo base usado: '11'.
Página 15 procesada. Prefijo base usado: '12'.
Página 16 procesada. Prefijo base usado: '13'.
Página 17 procesada. Prefijo base usado: '14'.
Página 18 procesada. Prefijo base usado: '15'.
Página 19 procesada. Pref

Extracción en alta calidad

In [8]:
import fitz  # PyMuPDF
import cv2
import numpy as np
import os

def ordenar_cajas(cajas, max_y_diff=50):
    if not cajas:
        return []
    cajas = sorted(cajas, key=lambda b: b[1])
    filas = []
    fila_actual = [cajas[0]]
    for caja in cajas[1:]:
        if abs(caja[1] - fila_actual[0][1]) < max_y_diff:
            fila_actual.append(caja)
        else:
            fila_actual = sorted(fila_actual, key=lambda b: b[0])
            filas.extend(fila_actual)
            fila_actual = [caja]
    fila_actual = sorted(fila_actual, key=lambda b: b[0])
    filas.extend(fila_actual)
    return filas

def extraer_imagenes_de_pdf(pdf_path, vector_nombres, output_dir="dataset_aislado", tamano_final=(400, 200)):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    doc = fitz.open(pdf_path)

    if len(vector_nombres) != len(doc):
        print(f"Aviso: El PDF tiene {len(doc)} páginas y el vector tiene {len(vector_nombres)} elementos.")

    contador_ceros = 0

    for num_pagina in range(min(len(doc), len(vector_nombres))):

        prefijo_base = vector_nombres[num_pagina]
        if prefijo_base == 0:
            contador_ceros += 1
            prefijo = f"00_{contador_ceros:02d}"
        else:
            prefijo = f"{prefijo_base:02d}"

        pagina = doc.load_page(num_pagina)

        # Resolución a 800 DPI
        pix = pagina.get_pixmap(dpi=800)

        img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.h, pix.w, pix.n)

        if pix.n == 4:
            img = cv2.cvtColor(img, cv2.COLOR_RGBA2BGR)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY_INV)

        kernel = np.ones((3, 3), np.uint8)
        thresh = cv2.dilate(thresh, kernel, iterations=1)

        contornos, _ = cv2.findContours(thresh, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)

        area_total_pagina = img.shape[0] * img.shape[1]
        cajas_brutas = []

        for c in contornos:
            x, y, w, h = cv2.boundingRect(c)
            area = w * h

            # Ajuste del área mínima proporcional a 800 DPI (12000 * 4)
            if 48000 < area < (area_total_pagina * 0.5):
                cajas_brutas.append((x, y, w, h))

        cajas_unicas = []
        for caja in cajas_brutas:
            x, y, w, h = caja
            es_duplicada = False
            for (ux, uy, uw, uh) in cajas_unicas:
                if abs(x - ux) < 30 and abs(y - uy) < 30:
                    es_duplicada = True
                    break
            if not es_duplicada:
                cajas_unicas.append(caja)

        cajas_unicas.sort(key=lambda b: b[2] * b[3], reverse=True)
        doce_cajas = cajas_unicas[:12]

        cajas_ordenadas = ordenar_cajas(doce_cajas)

        MARGEN_BORDE = 20

        for i, (x, y, w, h) in enumerate(cajas_ordenadas):
            if h > 2 * MARGEN_BORDE and w > 2 * MARGEN_BORDE:
                recorte = img[y + MARGEN_BORDE : y + h - MARGEN_BORDE, x + MARGEN_BORDE : x + w - MARGEN_BORDE]
            else:
                recorte = img[y:y+h, x:x+w]

            recorte_gris = cv2.cvtColor(recorte, cv2.COLOR_BGR2GRAY)
            recorte_suavizado = cv2.GaussianBlur(recorte_gris, (5, 5), 0)
            _, recorte_limpio = cv2.threshold(recorte_suavizado, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

            recorte_invertido = cv2.bitwise_not(recorte_limpio)

            # --- FILTRO PARA ELIMINAR PUNTOS SUELTOS (RUIDO) ---
            contornos_firma, _ = cv2.findContours(recorte_invertido, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

            for contorno_f in contornos_firma:
                # Modificado a 10 píxeles. Si el ruido persiste, sube este valor a 20 o 30.
                if cv2.contourArea(contorno_f) < 100:
                    cv2.drawContours(recorte_invertido, [contorno_f], -1, 0, -1)

            recorte_limpio = cv2.bitwise_not(recorte_invertido)
            # ----------------------------------------------------------

            puntos_tinta = cv2.findNonZero(recorte_invertido)

            if puntos_tinta is not None:
                x_f, y_f, w_f, h_f = cv2.boundingRect(puntos_tinta)
                firma_ajustada = recorte_limpio[y_f:y_f+h_f, x_f:x_f+w_f]
                firma_con_margen = cv2.copyMakeBorder(firma_ajustada, 10, 10, 10, 10, cv2.BORDER_CONSTANT, value=255)
            else:
                firma_con_margen = recorte_limpio

            h_actual, w_actual = firma_con_margen.shape
            w_final, h_final = tamano_final

            if h_actual == 0 or w_actual == 0:
                continue

            escala = min(w_final / w_actual, h_final / h_actual)
            nuevo_w = int(w_actual * escala)
            nuevo_h = int(h_actual * escala)

            firma_redimensionada = cv2.resize(firma_con_margen, (nuevo_w, nuevo_h), interpolation=cv2.INTER_AREA)
            lienzo_blanco = np.ones((h_final, w_final), dtype=np.uint8) * 255

            x_offset = (w_final - nuevo_w) // 2
            y_offset = (h_final - nuevo_h) // 2
            lienzo_blanco[y_offset:y_offset+nuevo_h, x_offset:x_offset+nuevo_w] = firma_redimensionada

            nombre_archivo = f"{num_pagina + 1:03d}_{prefijo}_{i + 1:02d}.png"

            ruta_archivo = os.path.join(output_dir, nombre_archivo)
            cv2.imwrite(ruta_archivo, lienzo_blanco)

        print(f"Página {num_pagina + 1} procesada. Prefijo base usado: '{prefijo}'.")

    print("¡Proceso completado!")

# --- EJECUCIÓN DEL SCRIPT ---
if __name__ == "__main__":
    from google.colab import drive
    drive.mount('/content/drive')
    ruta_al_pdf = "/content/drive/MyDrive/CUARTO/TFG VICTOR MARCOS/DATASET/DATASET.pdf"

    vector_de_nombres =[0,1,0,0,2,3,4,5,6,7,8,9,10,11,12,13,14,15,0,0,0,0,0,16,17,0,0,0,18,19,20,21,22,23,24,25,0,0,0,0,0,0,0,0,0,0,26,27,28,0,0,0,0,29,0,0,30,0,0,0,0,0,0,31,32,33,34,35,36,37,38,39,40,41,42,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]

    TAMAÑO_UNIFORME = (3200, 2400)

    extraer_imagenes_de_pdf(
        pdf_path=ruta_al_pdf,
        vector_nombres=vector_de_nombres,
        output_dir="IMAGENES_HIGH",
        tamano_final=TAMAÑO_UNIFORME
    )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Página 1 procesada. Prefijo base usado: '00_01'.
Página 2 procesada. Prefijo base usado: '01'.
Página 3 procesada. Prefijo base usado: '00_02'.
Página 4 procesada. Prefijo base usado: '00_03'.
Página 5 procesada. Prefijo base usado: '02'.
Página 6 procesada. Prefijo base usado: '03'.
Página 7 procesada. Prefijo base usado: '04'.
Página 8 procesada. Prefijo base usado: '05'.
Página 9 procesada. Prefijo base usado: '06'.
Página 10 procesada. Prefijo base usado: '07'.
Página 11 procesada. Prefijo base usado: '08'.
Página 12 procesada. Prefijo base usado: '09'.
Página 13 procesada. Prefijo base usado: '10'.
Página 14 procesada. Prefijo base usado: '11'.
Página 15 procesada. Prefijo base usado: '12'.
Página 16 procesada. Prefijo base usado: '13'.
Página 17 procesada. Prefijo base usado: '14'.
Página 18 procesada. Prefijo base usado: '15'.
Página 19 procesada. Pref

Redimensionado final e inversión de colores

In [13]:
import os
from PIL import Image, ImageOps

def procesar_imagenes(carpeta_entrada, carpeta_salida):

    # Extensiones de imagen válidas
    extensiones_validas = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')

    # Recorrer todos los archivos de la carpeta
    for nombre_archivo in os.listdir(carpeta_entrada):
        if nombre_archivo.lower().endswith(extensiones_validas):
            ruta_entrada = os.path.join(carpeta_entrada, nombre_archivo)
            ruta_salida = os.path.join(carpeta_salida, nombre_archivo)

            try:
                imagen = Image.open(ruta_entrada)

                if imagen.mode != 'L':
                    imagen = imagen.convert('L')

                imagen_redimensionada = imagen.resize((320, 240))

                imagen_invertida = ImageOps.invert(imagen_redimensionada)

                imagen_invertida.save(ruta_salida)
                print(f"Procesada con éxito: {nombre_archivo}")

            except Exception as e:
                print(f"Error al procesar la imagen {nombre_archivo}: {e}")

from google.colab import drive
drive.mount('/content/drive')

carpeta_origen = r'/content/drive/MyDrive/CUARTO/TFG VICTOR MARCOS/DATASET/IMAGENES_HIGH'
carpeta_destino = r'/content/drive/MyDrive/CUARTO/TFG VICTOR MARCOS/DATASET/IMAGENES_DEF'

procesar_imagenes(carpeta_origen, carpeta_destino)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Procesada con éxito: 044_00_19_10.png
Procesada con éxito: 044_00_19_11.png
Procesada con éxito: 001_00_01_01.png
Procesada con éxito: 001_00_01_02.png
Procesada con éxito: 001_00_01_03.png
Procesada con éxito: 001_00_01_04.png
Procesada con éxito: 001_00_01_05.png
Procesada con éxito: 001_00_01_06.png
Procesada con éxito: 001_00_01_07.png
Procesada con éxito: 001_00_01_08.png
Procesada con éxito: 001_00_01_09.png
Procesada con éxito: 001_00_01_10.png
Procesada con éxito: 001_00_01_11.png
Procesada con éxito: 001_00_01_12.png
Procesada con éxito: 002_01_01.png
Procesada con éxito: 002_01_02.png
Procesada con éxito: 002_01_03.png
Procesada con éxito: 002_01_04.png
Procesada con éxito: 002_01_05.png
Procesada con éxito: 002_01_06.png
Procesada con éxito: 002_01_07.png
Procesada con éxito: 002_01_08.png
Procesada con éxito: 002_01_09.png
Procesada con éxito: 002

Guardado de las carpetas en Google Drive

In [9]:
#!cp -r "/content/IMAGENES_LOW" "/content/drive/MyDrive/"
!cp -r "/content/IMAGENES_HIGH" "/content/drive/MyDrive/"